# Day 14 — Framing Messages Over a Pipe, Visualized

A pipe is a raw byte stream — no message boundaries. Prefix each message with its length
and the receiver can always find where one ends and the next begins.

In [ ]:
import struct

def frame(payload): return struct.pack('>I', len(payload)) + payload
def parse(buf):
    msgs, i = [], 0
    while len(buf) - i >= 4:
        (ln,) = struct.unpack('>I', buf[i:i+4])
        if len(buf) - i - 4 < ln: break
        msgs.append(buf[i+4:i+4+ln]); i += 4 + ln
    return msgs, buf[i:]

stream = frame(b'hello') + frame(b'world')
print('raw bytes:', stream)
print('parsed:   ', parse(stream)[0])

## Send framed messages through a real `os.pipe()` between two threads

In [ ]:
import os, threading

r, w = os.pipe()
messages = [b'SET x 1', b'GET x', b'KEYS']

def sender():
    for m in messages: os.write(w, frame(m))
    os.close(w)

received = []
def receiver():
    buf = b''
    while True:
        chunk = os.read(r, 3)      # tiny reads: frames arrive split up
        if not chunk: break
        buf += chunk
        msgs, buf = parse(buf)
        received.extend(msgs)
    os.close(r)

ts = threading.Thread(target=sender), threading.Thread(target=receiver)
ts[0].start(); ts[1].start(); ts[0].join(); ts[1].join()
print('received:', received)

## Takeaways

- Byte streams have no message boundaries; length-prefix framing restores them.
- The parser holds an incomplete frame as *leftover* until the rest of the bytes arrive.
- Pipes, sockets, and files are all 'a stream of bytes behind a file descriptor'.

**Now build** frame / parse_frames / KVStore / serve in `homework.py`, then `pytest -q`.